In [ ]:
# Configuración inicial e imports
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurar DATA_DIR
DATA_DIR = str(Path().resolve().parent)  # notebooks/ -> raíz
os.environ['DATA_DIR'] = DATA_DIR

print('='*70)
print('ANÁLISIS DE SATISFACCIÓN LABORAL - ENTORNO LOCAL')
print('='*70)
print(f'DATA_DIR: {DATA_DIR}')

# Verificar archivo principal
input_file = Path(DATA_DIR) / 'combined_with_replaced_categories.csv'
if not input_file.exists():
    print(f'\n❌ ERROR: No encontré {input_file}')
    print('Ejecuta primero los bloques de limpieza y categorización.')
    raise SystemExit()

print(f'✅ Archivo encontrado: {input_file.name}')
print('\n✅ Configuración completada')

In [ ]:
# =============================================================
# BLOQUE 5: Random Forest - Predicción de SATISFACCIÓN CON INGRESOS
# Target: "¿Qué tan conforme estás con tus ingresos laborales?" (1-2 = mal pago, 3-4 = bien pago)
# =============================================================

import os, re, unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

RANDOM_STATE = 42
N_ESTIMATORS = 150

DATA_DIR = os.environ.get("DATA_DIR") or str(Path().resolve().parent)
print("="*70)
print("RANDOM FOREST: PREDICCIÓN DE SATISFACCIÓN CON INGRESOS")
print("="*70)

# Cargar CSV
input_path = Path(DATA_DIR) / "combined_with_replaced_categories.csv"
df = pd.read_csv(input_path, encoding="utf-8-sig", dtype=object)
print(f"Datos cargados: {df.shape}")

# Normalizar nombres de columnas
def normalize_col(s: str) -> str:
    s = str(s) if s is not None else ""
    s = s.strip().lower()
    s = ''.join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    s = re.sub(r"[^0-9a-z]+", "_", s)
    return s.strip("_")

df.columns = [normalize_col(c) for c in df.columns]

# Target: satisfacción con ingresos (1-2 = mal pago, 3-4 = bien pago)
target_col = "que_tan_conforme_estas_con_tus_ingresos_laborales"
if target_col not in df.columns:
    raise SystemExit(f"ERROR: No encontré columna '{target_col}'")

y_raw = df[target_col].fillna("MISSING")

def map_conformidad(v):
    """Mapea respuesta a: 1_2 (mal pago) o 3_4 (bien pago)"""
    s = str(v).strip()
    m = re.search(r"([1-4])", s)
    if not m:
        return "OTHER"
    n = int(m.group(1))
    return "1_2" if n in (1,2) else "3_4"

y = y_raw.apply(map_conformidad)

print("\n📊 Distribución del target:")
print(y.value_counts())
print(f"\n  • '1_2' = Mal pago / Insatisfecho")
print(f"  • '3_4' = Bien pago / Satisfecho")

# Codificar target
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Features
desired_features = [
    'tengo_edad',
    'anos_de_experiencia',
    'anos_en_el_puesto_actual',
    'cuantas_personas_tenes_a_cargo',
    'cantidad_de_personas_en_tu_organizacion',
    'trabajo_de',
    'sueldo_bruto_en_dolares',
    'dedicacion',
    'recibio_actualizacion_de_ingresos',
    'de_que_fue_el_ajuste_total_acumulado',
]

features_ok = sorted(set(c for c in desired_features if c in df.columns))
print(f"\n✅ Usando {len(features_ok)} features")

X = df[features_ok].copy()

# Detectar numéricas/categóricas
def is_numeric(col: pd.Series) -> bool:
    cleaned = col.astype(str).str.replace(r"[\$€£\s]", "", regex=True).str.replace(r",(?=\d{1,2}$)", ".", regex=True)
    numeric_try = pd.to_numeric(cleaned, errors="coerce")
    return numeric_try.notna().mean() >= 0.30

numeric_cols = [c for c in features_ok if is_numeric(X[c])]
categorical_cols = [c for c in features_ok if c not in numeric_cols]

print(f"  • Numéricas: {len(numeric_cols)}")
print(f"  • Categóricas: {len(categorical_cols)}")

# Pipeline
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_cols),
    ],
    remainder="drop",
    sparse_threshold=0
)

pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("rf", RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1))
])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.20, random_state=RANDOM_STATE, stratify=y_enc
)
print(f"\nSplit: Train={X_train.shape[0]}, Test={X_test.shape[0]}")

# Entrenar
print("\nEntrenando Random Forest...")
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

# Métricas
acc = accuracy_score(y_test, y_pred)
print(f"\n✅ Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(pd.DataFrame(cm, index=le.classes_, columns=le.classes_))

# Guardar modelo
model_path = Path(DATA_DIR) / "backend" / "rf_satisfaccion_ingresos.pkl"
model_path.parent.mkdir(exist_ok=True)
joblib.dump(pipe, model_path)
print(f"\n💾 Modelo guardado: {model_path}")

# Feature Importance
rf = pipe.named_steps["rf"]
feat_order = numeric_cols + categorical_cols
fi = pd.DataFrame({"feature": feat_order, "importance": rf.feature_importances_})
fi = fi.sort_values("importance", ascending=False)
print("\n📊 Top 10 Features Más Importantes:")
print(fi.head(10).to_string(index=False))

print("\n✅ Random Forest completado")

In [ ]:
# =============================================================
# BLOQUE 9: XGBoost - Predicción de BÚSQUEDA DE TRABAJO
# Target: "estas_buscando_trabajo" (Sí vs No)
# =============================================================

import os, re, unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except ImportError:
    print('Instalando XGBoost...')
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])
    import xgboost as xgb

DATA_DIR = os.environ.get('DATA_DIR') or str(Path().resolve().parent)
print('='*70)
print('XGBOOST: PREDICCIÓN DE BÚSQUEDA DE TRABAJO')
print('='*70)

# Cargar datos
input_path = Path(DATA_DIR) / 'combined_with_replaced_categories.csv'
df = pd.read_csv(input_path, encoding='utf-8-sig', dtype=object)
print(f"Datos cargados: {df.shape}")

# Normalizar columnas
def normalize_col(s: str) -> str:
    s = str(s).strip().lower()
    s = ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
    return re.sub(r'[^0-9a-z]+', '_', s).strip('_')

df.columns = [normalize_col(c) for c in df.columns]

# Target
target_col = 'estas_buscando_trabajo'
if target_col not in df.columns:
    raise SystemExit(f"ERROR: No encontré '{target_col}'")

print(f"Target: {target_col}")

# Mapeo
def map_buscando_trabajo(v):
    if pd.isna(v):
        return 'MISSING'
    s = str(v).strip()
    if 'activamente' in s.lower():
        return 'Si'
    if 'conforme' in s.lower() or 'ofertas' in s.lower():
        return 'No'
    return 'OTHER'

y = df[target_col].apply(map_buscando_trabajo)

print("\n📊 Distribución:")
print(y.value_counts())

# Filtrar clases válidas
valid = ['Si', 'No']
mask = y.isin(valid)
df = df[mask].reset_index(drop=True)
y = y[mask].reset_index(drop=True)

print(f"\nFilas finales: {len(df)} (Si: {(y=='Si').sum()}, No: {(y=='No').sum()})")

# Features
desired_features = [
    'tengo_edad',
    'anos_de_experiencia',
    'anos_en_el_puesto_actual',
    'cuantas_personas_tenes_a_cargo',
    'cantidad_de_personas_en_tu_organizacion',
    'trabajo_de',
    'sueldo_bruto_en_dolares',
    'estudios_estado',
    'dedicacion',
    'recibio_actualizacion_de_ingresos',
    'de_que_fue_el_ajuste_total_acumulado',
]

X = df[[c for c in desired_features if c in df.columns]].copy()
print(f"\n✅ Usando {len(X.columns)} features")

# Convertir a numérico
for c in X.columns:
    coer = pd.to_numeric(
        X[c].astype(str).str.replace(r'[$€£%\s]', '', regex=True).str.replace(r',(?=\d{1,2}$)', '.', regex=True),
        errors='coerce'
    )
    if coer.notna().sum() >= max(10, int(0.02 * len(df))):
        X[c] = coer.fillna(-1.0)
    else:
        X[c] = pd.factorize(X[c].fillna('MISSING').astype(str))[0]

# Encode target
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print(f"Split: Train={X_train.shape[0]}, Test={X_test.shape[0]}")

# XGBoost con GridSearch
xgb_base = xgb.XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')

param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [4, 6],
    'learning_rate': [0.1],
}

print("\nEntrenando XGBoost con GridSearchCV...")
grid_search = GridSearchCV(xgb_base, param_grid, cv=2, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f"\n✅ Mejores parámetros: {grid_search.best_params_}")
print(f"F1 Score (CV): {grid_search.best_score_:.4f}")

# Predicción
xgb_model = grid_search.best_estimator_
y_pred = xgb_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\n✅ Accuracy: {acc:.4f}")
print(f"✅ F1 Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Guardar modelo
model_path = Path(DATA_DIR) / "backend" / "xgb_busqueda_trabajo.pkl"
joblib.dump(xgb_model, model_path)
print(f"\n💾 Modelo guardado: {model_path}")

print("\n✅ XGBoost completado")

In [ ]:
# =============================================================
# BLOQUE COMPLETO: Clustering KMeans + PCA + GRAFICOS
# (Incluye One-Hot para categóricas, PCA, búsqueda de k y gráficos)
# =============================================================

import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------
# Parámetros y paths
# -----------------------
DATA_DIR = os.environ.get('DATA_DIR') or str(Path().resolve())
print('Usando DATA_DIR =', DATA_DIR)

input_path = Path(DATA_DIR) / 'combined_with_replaced_categories.csv'
if not input_path.exists():
    raise SystemExit('No encontré el CSV combinado en DATA_DIR.')

print('Cargando datos desde:', input_path)
df = pd.read_csv(input_path, encoding='utf-8-sig', dtype=object)
print('Shape raw:', df.shape)

# -----------------------
# Filtrado por género (igual que antes)
# -----------------------
def normalize_col_local(s: str) -> str:
    import re, unicodedata
    if s is None: return ''
    s = str(s).strip().lower()
    s = ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
    s = re.sub(r'[^0-9a-z]+', '_', s)
    s = re.sub(r'_+', '_', s)
    return s.strip('_')

gen_col_candidates = [c for c in df.columns if 'gener' in normalize_col_local(c)]
gen_col = gen_col_candidates[0] if gen_col_candidates else None
if gen_col:
    def map_genero(v):
        if pd.isna(v): return 'MISSING'
        s = ''.join(c for c in __import__('unicodedata').normalize('NFKD', str(v).strip().lower())
                    if not __import__('unicodedata').combining(c))
        if 'hombre' in s: return 'Hombre'
        if 'mujer' in s: return 'Mujer'
        return 'MISSING'
    df['__gen_map'] = df[gen_col].apply(map_genero)
    kept_mask = df['__gen_map'].isin(['Hombre', 'Mujer'])
    print(f"Filtrado por género: {kept_mask.sum()} filas restantes.")
    df = df[kept_mask].reset_index(drop=True)
else:
    print("No se detectó columna de género. Se continúa sin filtrar.")

# -----------------------
# Variables seleccionadas (igual que antes)
# -----------------------
selected_cols = [
    'Años de experiencia',
    'antiguedad_en_la_empresa_actual',
    'sueldo_bruto_en_dolares',
    'Trabajo de'
]
df = df[[c for c in selected_cols if c in df.columns]].copy()
print('Columnas usadas:', df.columns.tolist())

# -----------------------
# Detectar numéricas y categóricas (igual que antes)
# -----------------------
numeric_cols = []
cat_cols = []
for c in df.columns:
    s = df[c].astype(str).str.replace(r'[\$€£\s]', '', regex=True)
    s = s.str.replace(r',(?=\d{1,2}$)', '.', regex=True)
    coer = pd.to_numeric(s.replace({'nan': None}), errors='coerce')
    if coer.notna().sum() >= max(10, int(0.02 * len(df))):
        numeric_cols.append(c)
    else:
        cat_cols.append(c)

print(f'Columnas numéricas detectadas: {numeric_cols}')
print(f'Columnas categóricas detectadas: {cat_cols}')

# -----------------------
# Procesamiento numéricas
# -----------------------
X_num = pd.DataFrame(index=df.index)
for c in numeric_cols:
    s = df[c].astype(str).str.replace(r'[\$€£\s]', '', regex=True)
    s = s.str.replace(r',(?=\d{1,2}$)', '.', regex=True)
    X_num[c] = pd.to_numeric(s.replace({'nan': None}), errors='coerce')
X_num = X_num.fillna(X_num.median())

# -----------------------
# One-Hot Encoding para categóricas
# -----------------------
if cat_cols:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_cat = encoder.fit_transform(df[cat_cols].astype(str))
    cat_feature_names = encoder.get_feature_names_out(cat_cols)
    X_cat = pd.DataFrame(X_cat, columns=cat_feature_names, index=df.index)
    X_full = pd.concat([X_num, X_cat], axis=1)
else:
    X_full = X_num.copy()

# FIX: asegurar que todas las columnas sean strings (evita error de sklearn)
X_full.columns = X_full.columns.astype(str)

# -----------------------
# Estandarización
# -----------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)

# -----------------------
# PCA - reducir para visualización y clustering dentro de un espacio compacto
# Intentamos tener al menos 2 componentes para el scatter; si hay menos, usar lo que haya.
# -----------------------
n_components = min(10, X_scaled.shape[1]) if X_scaled.shape[1] > 1 else 1
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)

# Para plotting 2D nos aseguramos de tener 2 componentes (si solo hay 1, duplicamos la columna para el scatter)
if X_pca.shape[1] == 1:
    X_pca = np.hstack([X_pca, np.zeros((X_pca.shape[0], 1))])

explained = pca.explained_variance_ratio_.sum()
print(f'PCA reducido a {X_pca.shape[1]} componentes (varianza explicada = {explained:.3f})')

# -----------------------
# KMeans + búsqueda de k óptimo por silhouette (k=2..10)
# -----------------------
best_k, best_score, best_labels = None, -1.0, None
for k in range(2, min(10, max(3, X_pca.shape[0]//2))):  # evitar k>n_samples/2 absurdo
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    try:
        score = silhouette_score(X_pca, labels)
    except Exception:
        score = -1.0
    print(f'k={k} → silhouette={score:.4f}')
    if score > best_score:
        best_score, best_k, best_labels = score, k, labels

if best_labels is None:
    # fallback: k=2
    km = KMeans(n_clusters=2, random_state=42, n_init=10)
    best_labels = km.fit_predict(X_pca)
    best_k = 2
    best_score = silhouette_score(X_pca, best_labels)

print(f'Mejor k={best_k} con silhouette={best_score:.4f}')

# -----------------------
# Guardar resultado en CSV (misma lógica que antes)
# -----------------------
df_out = df.copy()
df_out['cluster_kmeans'] = (best_labels + 1) if best_labels is not None else 0
out_path = Path(DATA_DIR) / 'combined_with_clusters.csv'
df_out.to_csv(out_path, index=False, encoding='utf-8-sig')
print('Guardado:', out_path)

# -----------------------
# Resumen por cluster (medianas)
# -----------------------
if best_labels is not None:
    summary_df = X_num.copy()
    summary_df['cluster_kmeans'] = best_labels + 1
    summary = summary_df.groupby('cluster_kmeans')[numeric_cols].median()
    print('\nResumen por cluster (medianas de variables numéricas):')
    print(summary)

# =============================================================
# ======================= GRAFICOS ============================
# =============================================================

# Rutas de salida
plot_scatter_path = Path(DATA_DIR) / "clusters_pca_scatter.png"
plot_centroids_path = Path(DATA_DIR) / "clusters_pca_with_centroids.png"
plot_heatmap_path = Path(DATA_DIR) / "clusters_heatmap.png"
plot_boxplots_path = Path(DATA_DIR) / "clusters_boxplots.png"

# --- Scatter PCA colores por cluster ---
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=best_labels,
    cmap='tab10',
    s=30,
    alpha=0.85,
    edgecolors='w',
    linewidth=0.3
)
plt.title("Clusters KMeans proyectados en PCA (2 componentes)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid(alpha=0.2)
cb = plt.colorbar(scatter, ticks=range(0, (best_k if best_k else np.max(best_labels)+1)))
cb.set_label('Cluster (0-based)')
plt.tight_layout()
plt.savefig(plot_scatter_path, dpi=200)
plt.close()
print("Scatter PCA guardado en:", plot_scatter_path)

# --- Scatter + centroides (en espacio PCA) ---
# calcular centroides en el espacio PCA usando los labels del mejor modelo KMeans recalculado
 #--- Scatter + centroides (proyectados correctamente a PCA2) ---
# Recalcular KMeans en el espacio PCA completo (10D o el que tengas)
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_final.fit(X_pca)

# Centroides en PCA full (por ej. 10D)
centroids_full = km_final.cluster_centers_

# PCA solo para graficar a 2D
pca_plot = PCA(n_components=2)
X_plot = pca_plot.fit_transform(X_pca)
centroids_2d = pca_plot.transform(centroids_full)

plt.figure(figsize=(10,7))
plt.scatter(X_plot[:,0], X_plot[:,1], c=best_labels,
            cmap='tab10', s=25, alpha=0.7, edgecolors='w', linewidth=0.2)

# centroides graficados en 2D
plt.scatter(centroids_2d[:,0], centroids_2d[:,1], c='black', s=200, marker='X')

for i, (cx, cy) in enumerate(centroids_2d):
    plt.text(cx, cy, f"  C{i+1}", color='black', fontsize=9, weight='bold')

plt.title("Clusters KMeans en PCA (2D) con centroides proyectados")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(plot_centroids_path, dpi=200)
plt.close()
print("Scatter con centroides guardado en:", plot_centroids_path)
plt.title("Clusters KMeans en PCA con centroides")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(plot_centroids_path, dpi=200)
plt.close()
print("Scatter con centroides guardado en:", plot_centroids_path)

# --- Heatmap de medianas por cluster (variables numéricas originales) ---
if best_labels is not None:
    # usamos summary_df calculado antes (medianas por cluster)
    cluster_medians = summary_df.groupby('cluster_kmeans')[numeric_cols].median()
    plt.figure(figsize=(max(6, len(numeric_cols)*1.5), 6))
    sns.heatmap(cluster_medians, annot=True, fmt=".2f", cmap="viridis")
    plt.title("Medianas de variables numéricas por cluster")
    plt.tight_layout()
    plt.savefig(plot_heatmap_path, dpi=200)
    plt.close()
    print("Heatmap guardado en:", plot_heatmap_path)
else:
    print("No hay labels para heatmap.")

# --- Boxplots por variable por cluster (una figura con subplots) ---
if best_labels is not None:
    n_vars = len(numeric_cols)
    ncols = 2 if n_vars > 1 else 1
    nrows = int(np.ceil(n_vars / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(6*ncols, 4*nrows))
    if n_vars == 1:
        axes = np.array([[axes]])
    axes = axes.flatten()
    for i, var in enumerate(numeric_cols):
        sns.boxplot(x=summary_df['cluster_kmeans'], y=summary_df[var], ax=axes[i])
        axes[i].set_title(f"{var} por cluster")
        axes[i].set_xlabel("cluster_kmeans")
        axes[i].set_ylabel(var)
    # eliminar ejes extra
    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])
    plt.tight_layout()
    plt.savefig(plot_boxplots_path, dpi=200)
    plt.close()
    print("Boxplots guardado en:", plot_boxplots_path)

print("\nTodos los gráficos generados y guardados en DATA_DIR.")


In [ ]:
# =============================================================
# BLOQUE 10: Regresión Lineal - Sueldo Actual (USD)
# =============================================================

import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib

DATA_DIR = os.environ.get('DATA_DIR') or str(Path().resolve().parent)
print('='*70)
print('REGRESIÓN LINEAL: SUELDO ACTUAL (USD)')
print('='*70)

# Cargar
input_path = Path(DATA_DIR) / 'combined_with_replaced_categories.csv'
df = pd.read_csv(input_path, encoding='utf-8-sig')
print(f"Datos: {df.shape}")

# Target
target_col = 'sueldo_neto_en_dolares'
if target_col not in df.columns:
    raise SystemExit(f"No encontré '{target_col}'")

df = df[df[target_col].notna() & (df[target_col] > 0)].reset_index(drop=True)
print(f"Filas válidas: {len(df)}")

# Features
selected_features = [
    "Años de experiencia",
    "antiguedad_en_la_empresa_actual",
    "Años en el puesto actual",
    "cuantas_personas_tenes_a_cargo",
    "Trabajo de",
    "seniority",
    "dedicacion",
    "tengo_edad",
]

existing_features = [c for c in selected_features if c in df.columns]
print(f"\nFeatures: {len(existing_features)}")

X = df[existing_features].copy()
y = df[target_col].copy()

# Convertir a numérico
for c in X.columns:
    coer = pd.to_numeric(X[c].astype(str).str.replace(r'[$€£\s]', '', regex=True), errors='coerce')
    if coer.notna().sum() >= max(10, int(0.02 * len(df))):
        X[c] = coer.fillna(coer.median())
    else:
        X[c] = pd.factorize(X[c].fillna('MISSING').astype(str))[0]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Split: Train={X_train.shape[0]}, Test={X_test.shape[0]}")

# Escalar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Entrenar
model = Ridge(alpha=1.0, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

# Métricas
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"\n✅ R² Score: {r2:.4f}")
print(f"✅ RMSE: ${rmse:.2f}")
print(f"✅ MAE: ${mae:.2f}")

# Guardar
model_path = Path(DATA_DIR) / "backend" / "reg_sueldo_actual_usd.pkl"
joblib.dump(model, model_path)
scaler_path = Path(DATA_DIR) / "backend" / "reg_sueldo_actual_usd_scaler.pkl"
joblib.dump(scaler, scaler_path)
print(f"\n💾 Modelo guardado: {model_path}")

print("\n✅ Regresión completada")

In [ ]:
# =============================================================
# BLOQUE 11: Regresión Lineal - Sueldo Futuro (Pesos)
# =============================================================

import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib

DATA_DIR = os.environ.get('DATA_DIR') or str(Path().resolve().parent)
print('='*70)
print('REGRESIÓN LINEAL: SUELDO FUTURO (Pesos ARS)')
print('='*70)

# Cargar
input_path = Path(DATA_DIR) / 'combined_with_replaced_categories.csv'
df = pd.read_csv(input_path, encoding='utf-8-sig')
print(f"Datos: {df.shape}")

# Verificar year
if 'year' not in df.columns:
    print("⚠️ No encontré columna 'year'. Saltando este análisis.")
else:
    # Target
    target_col = 'ultimo_salario_mensual_o_retiro_neto_en_pesos_argentinos'
    if target_col not in df.columns:
        raise SystemExit(f"No encontré '{target_col}'")

    df = df[df[target_col].notna() & (df[target_col] > 0) & df['year'].notna()].reset_index(drop=True)
    print(f"Filas válidas: {len(df)}")
    print(f"Años: {sorted(df['year'].unique())}")

    # Features (incluye 'year' para tendencia temporal)
    selected_features = [
        "year",
        "Años de experiencia",
        "antiguedad_en_la_empresa_actual",
        "cuantas_personas_tenes_a_cargo",
        "Trabajo de",
        "seniority",
        "tengo_edad",
    ]

    existing_features = [c for c in selected_features if c in df.columns]
    print(f"\nFeatures: {len(existing_features)} (incluye 'year' ⭐)")

    X = df[existing_features].copy()
    y = df[target_col].copy()

    # Convertir
    for c in X.columns:
        if c == 'year':
            X[c] = pd.to_numeric(X[c], errors='coerce').fillna(2024)
            continue
        coer = pd.to_numeric(X[c].astype(str).str.replace(r'[$€£%\s]', '', regex=True), errors='coerce')
        if coer.notna().sum() >= max(10, int(0.02 * len(df))):
            X[c] = coer.fillna(coer.median())
        else:
            X[c] = pd.factorize(X[c].fillna('MISSING').astype(str))[0]

    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Split: Train={X_train.shape[0]}, Test={X_test.shape[0]}")

    # Escalar
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Entrenar
    model = Ridge(alpha=1.0, random_state=42)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    # Métricas
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    print(f"\n✅ R² Score: {r2:.4f}")
    print(f"✅ RMSE: ${rmse:,.2f} ARS")
    print(f"✅ MAE: ${mae:,.2f} ARS")

    # Guardar
    model_path = Path(DATA_DIR) / "backend" / "reg_sueldo_futuro_ars.pkl"
    joblib.dump(model, model_path)
    scaler_path = Path(DATA_DIR) / "backend" / "reg_sueldo_futuro_ars_scaler.pkl"
    joblib.dump(scaler, scaler_path)
    print(f"\n💾 Modelo guardado: {model_path}")

    print("\n✅ Regresión completada")

# ✅ Análisis Completado

## Modelos Generados:

1. **Random Forest** → `backend/rf_satisfaccion_ingresos.pkl`
   - Predice: Satisfacción con ingresos (1-2 mal / 3-4 bien)

2. **XGBoost** → `backend/xgb_busqueda_trabajo.pkl`
   - Predice: Búsqueda de trabajo (Sí / No)

3. **Clustering** → `combined_with_clusters.csv`
   - Segmentación de empleados en grupos

4. **Regresión USD** → `backend/reg_sueldo_actual_usd.pkl`
   - Predice: Sueldo actual en dólares

5. **Regresión ARS** → `backend/reg_sueldo_futuro_ars.pkl`
   - Predice: Sueldo futuro en pesos

## Próximos pasos:

1. Ejecutar backend: `cd backend && python app.py`
2. Ejecutar frontend: `cd frontend && python server.py`
3. Abrir: `http://localhost:8000`